## Configuration and Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer

# --- CONFIGURATION ---
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
DATASET_PATH = "medalpaca/medical_meadow_medical_flashcards"
DATASET_SPLIT = "train"
SAMPLE_SIZE = 5000
RANDOM_SEED = 42
SYSTEM_PROMPT = "You are a professional medical assistant. Provide accurate, concise, and helpful medical information."
MEDICAL_KEYWORDS = [
    "levels",
    "syndrome",
    "deficiency",
    "suggests",
    "treatment",
    "disease",
    "receptor",
]


## EDA

In [ ]:
def run_medical_meadow_flashcards_eda():
    print("1. Loading Medical Dataset and Tokenizer...")
    dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)
    df = dataset.to_pandas()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    print("\n2. Analyzing Column Roles & Structural Redundancy...")
    total_rows = len(df)
    unique_instructions = df["instruction"].nunique()

    print(f"Total Rows: {total_rows}")
    print(f"Unique Instructions: {unique_instructions}")
    print("\nTop 5 Instructions (Task Prompts):")
    print(df["instruction"].value_counts().head(5))

    empty_inputs = (df["input"].str.strip() == "").sum()
    print(f"\nRows with empty 'input': {empty_inputs}")

    print("\n3. Completion (Answer) Content Integrity...")
    df["has_med_signal"] = df["output"].apply(
        lambda x: any(kw in str(x).lower() for kw in MEDICAL_KEYWORDS)
    )
    print(
        f"Rows with basic medical terminology signal: {df['has_med_signal'].sum()} ({(df['has_med_signal'].sum() / total_rows) * 100:.2f}%)"
    )

    print("\n4. Token Length Distributions (The 'Whole Thing')...")

    def calculate_full_lengths(row):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"{row['instruction']}\n\nQuestion: {row['input']}",
            },
            {"role": "assistant", "content": str(row["output"])},
        ]

        full_text = tokenizer.apply_chat_template(messages, tokenize=False)
        total_tokens = len(tokenizer.encode(full_text))

        prompt_text = tokenizer.apply_chat_template(messages[:2], tokenize=False)
        prompt_tokens = len(tokenizer.encode(prompt_text))

        return pd.Series([prompt_tokens, (total_tokens - prompt_tokens), total_tokens])

    sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).copy()
    sample_df[["prompt_tokens", "completion_tokens", "total_tokens"]] = sample_df.apply(
        calculate_full_lengths, axis=1
    )

    print("\nFull Sequence Percentiles (Llama 3 Tokens):")
    print(
        sample_df[["prompt_tokens", "completion_tokens", "total_tokens"]].describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    print("\n5. Visualizing Medical Context Usage...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.histplot(
        sample_df["prompt_tokens"], bins=30, ax=axes[0], color="blue"
    ).set_title("Medical Prompt Length")
    sns.histplot(
        sample_df["completion_tokens"], bins=30, ax=axes[1], color="orange"
    ).set_title("Medical Answer Length")
    sns.histplot(
        sample_df["total_tokens"], bins=30, ax=axes[2], color="green"
    ).set_title("Total Sequence Length")
    plt.show()

    print("\n6. Medical Vocabulary (Router Prep)...")
    vectorizer = CountVectorizer(stop_words="english", max_features=25)
    word_counts = vectorizer.fit_transform(df["input"].astype(str)).sum(axis=0)
    words_freq = [
        (word, word_counts[0, idx]) for word, idx in vectorizer.vocabulary_.items()
    ]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    words, counts = zip(*words_freq)
    plt.figure(figsize=(10, 8))
    sns.barplot(x=list(counts), y=list(words), palette="magma")
    plt.title("Top 25 Diagnostic Keywords in Medical Input")
    plt.show()


## Execution

In [ ]:
if __name__ == "__main__":
    run_medical_meadow_flashcards_eda()
